In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import (
    col, coalesce, when, trim, initcap, lower,
    to_date, regexp_replace, from_json, get_json_object,
    nanvl, lit, upper
)
from pyspark.sql.types import StringType, MapType, DoubleType


In [0]:

US_STATES = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
    "Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa",
    "Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan",
    "Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada",
    "New Hampshire","New Jersey","New Mexico","New York","North Carolina",
    "North Dakota","Ohio","Oklahoma","Oregon","Pennsylvania","Rhode Island",
    "South Carolina","South Dakota","Tennessee","Texas","Utah","Vermont",
    "Virginia","Washington","West Virginia","Wisconsin","Wyoming"
]

@dp.temporary_view(
    name="customers_bronze_view",
    comment="Standardised Bronze customers: unified column names, R4 city/state swap fixed, segment/region normalised."
)
def customers_bronze_view():
    df = spark.readStream.table("customers").select(
        # Unify the four ID column variants into one canonical column
        coalesce(
            col("customer_id"),
            col("CustomerID"),
            col("cust_id"),
            col("customer_identifier")
        ).alias("customer_id"),

        # Unify the two email column variants
        coalesce(
            col("customer_email"),
            col("email_address")
        ).alias("customer_email"),

        # Unify the two name column variants
        coalesce(
            col("customer_name"),
            col("full_name")
        ).alias("customer_name"),

        # Unify the two segment column variants (raw — normalised below)
        coalesce(
            col("segment"),
            col("customer_segment")
        ).alias("segment"),

        col("country"),
        col("city"),
        col("state"),
        col("postal_code"),
        col("region"),
        col("source_region"),
        col("_rescued_data"),
        col("_source_file_path"),
        col("_source_file_name"),
        col("_source_modified_time"),
        col("_ingested_at"),
    )

    # ── R4 city/state swap fix ─────────────────────────────────────────────
    # Region 4 has the city and state columns physically interchanged in the
    # source file. We detect this by checking whether the city column contains
    # a value that is actually a US state name. When it does, swap them back.
    city_is_state = col("city").isin(US_STATES)
    df = df.withColumn("city_fixed",
        when(city_is_state, col("state")).otherwise(col("city"))
    ).withColumn("state_fixed",
        when(city_is_state, col("city")).otherwise(col("state"))
    ).drop("city", "state") \
     .withColumnRenamed("city_fixed", "city") \
     .withColumnRenamed("state_fixed", "state")

    # ── Segment normalisation ──────────────────────────────────────────────
    # Abbreviations and typos found during Bronze analysis mapped to the three
    # canonical values that the Silver expectation rule allows.
    df = df.withColumn("segment",
        when(col("segment").isin("CONS", "Consumer", "Cosumer"), "Consumer")
       .when(col("segment").isin("CORP", "Corporate"),           "Corporate")
       .when(col("segment").isin("HO", "Home Office"),           "Home Office")
       .otherwise(col("segment"))   # unknown values surface in quarantine
    )

    # ── Region normalisation ───────────────────────────────────────────────
    df = df.withColumn("region",
        when(col("region").isin("N", "North"),    "North")
       .when(col("region").isin("S", "South"),    "South")
       .when(col("region").isin("E", "East"),     "East")
       .when(col("region").isin("W", "West"),     "West")
       .when(col("region").isin("C", "Central"),  "Central")
       .otherwise(col("region"))
    )

    return df


In [0]:
@dp.temporary_view(
    name="orders_bronze_view",
    comment="Standardised Bronze orders: both date formats parsed, status and ship_mode normalised."
)
def orders_bronze_view():
    df = spark.readStream.table("orders")

    # ── Date parsing — handle both MM/DD/YYYY HH:mm (R1) and YYYY-MM-DD (R3, R5)
    # coalesce tries the first format; if it returns null (wrong format), tries the second.
    # Any date that fails both formats becomes null and is caught by the quarantine rule.
    date_cols = {
        "order_purchase_date":           ("MM/dd/yyyy HH:mm", "yyyy-MM-dd"),
        "order_approved_at":             ("MM/dd/yyyy HH:mm", "yyyy-MM-dd"),
        "order_delivered_carrier_date":  ("MM/dd/yyyy HH:mm", "yyyy-MM-dd"),
        "order_delivered_customer_date": ("MM/dd/yyyy HH:mm", "yyyy-MM-dd"),
        "order_estimated_delivery_date": ("MM/dd/yyyy HH:mm", "yyyy-MM-dd"),
    }
    for col_name, (fmt1, fmt2) in date_cols.items():
        df = df.withColumn(col_name,
            coalesce(
                to_date(col(col_name), fmt1),
                to_date(col(col_name), fmt2),
            )
        )

    # ── order_status normalisation ─────────────────────────────────────────
    df = df.withColumn("order_status",
        when(col("order_status").isin("canceled",    "Cancelled"),   "Cancelled")
       .when(col("order_status").isin("created",     "Created"),     "Created")
       .when(col("order_status").isin("delivered",   "Delivered"),   "Delivered")
       .when(col("order_status").isin("invoiced",    "Invoiced"),    "Invoiced")
       .when(col("order_status").isin("processing",  "Processing"),  "Processing")
       .when(col("order_status").isin("shipped",     "Shipped"),     "Shipped")
       .when(col("order_status").isin("unavailable", "Unavailable"), "Unavailable")
       .otherwise(col("order_status"))
    )

    # ── ship_mode normalisation ────────────────────────────────────────────
    df = df.withColumn("ship_mode",
        when(col("ship_mode") == "1st Class",  "First Class")
       .when(col("ship_mode") == "2nd Class",  "Second Class")
       .when(col("ship_mode") == "Std Class",  "Standard Class")
       .otherwise(initcap(trim(col("ship_mode"))))
    )

    return df

In [0]:
@dp.temporary_view(
    name="returns_bronze_view",
    comment="Standardised Bronze returns: schema merged, rescued refund_amount recovered, NaN→null, dates parsed, garbage reasons nulled."
)
def returns_bronze_view():
    df = spark.readStream.table("returns")

    # ── Step 1: Recover refund_amount from _rescued_data (R6 "$" string) ──
    # R6 stored refund_amount as "$130.65". The "$" caused Auto Loader to rescue
    # these rows. We parse the JSON, strip "$", and cast to DOUBLE before COALESCE
    # so the recovered value takes priority over the null in the refund_amount column.
    df = df.withColumn(
        "refund_amount_rescued",
        regexp_replace(
            get_json_object(col("_rescued_data"), "$.refund_amount"),
            r"[$,]", ""
        ).cast(DoubleType())
    )

    # ── Step 2: Unify column name variants into canonical names ───────────
    df = df.select(
        coalesce(col("order_id"),      col("OrderId")).alias("order_id"),
        coalesce(col("return_reason"), col("reason")).alias("return_reason"),
        coalesce(col("return_date"),   col("date_of_return")).alias("return_date"),
        # Prefer the rescued double value, then the raw refund_amount (R6 non-rescued rows),
        # then fall back to the amount column (returns_2.json format)
        coalesce(
            col("refund_amount_rescued"),
            nanvl(col("refund_amount"), lit(None).cast(DoubleType())),
            col("amount")
        ).alias("refund_amount"),
        coalesce(col("return_status"), col("status")).alias("return_status"),
        col("_rescued_data"),
        col("_source_file_path"),
        col("_source_file_name"),
        col("_source_modified_time"),
        col("_ingested_at"),
    )

    # ── Step 3: Parse return_date — handle "NULL" string and two date formats
    # First null out the literal "NULL" string so to_date does not choke on it.
    # Then try YYYY-MM-DD (ISO, returns_2.json) then DD-MM-YYYY (returns_2.json variant).
    df = df.withColumn("return_date",
        when(upper(trim(col("return_date"))) == "NULL", lit(None))
        .otherwise(col("return_date"))
    )
    df = df.withColumn("return_date",
        coalesce(
            to_date(col("return_date"), "yyyy-MM-dd"),
            to_date(col("return_date"), "dd-MM-yyyy"),
            to_date(col("return_date"), "MM/dd/yyyy"),
        )
    )

    # ── Step 4: Garbage reason values → SQL null ──────────────────────────
    # "?" and the literal string "NULL" both mean unknown — treat as null
    # so Silver rules and Gold aggregations handle them consistently.
    df = df.withColumn("return_reason",
        when(
            col("return_reason").isNull() |
            (trim(col("return_reason")) == "?") |
            (upper(trim(col("return_reason"))) == "NULL"),
            lit(None)
        ).otherwise(trim(col("return_reason")))
    )

    return df

In [0]:
@dp.temporary_view(
    name="transactions_bronze_view",
    comment="Standardised Bronze transactions: rescued Order_ID/Product_ID recovered, discount normalised, types cast."
)
def transactions_bronze_view():
    df = spark.readStream.table("transactions")

    # ── Step 1: Recover Order_ID and Product_ID from _rescued_data (R3) ──
    # R3 used "Order_ID" (capital D) but schema inferred "Order_id" (lowercase d).
    # Auto Loader rescued those rows. We extract both keys from the JSON.
    df = df.withColumn("rescued_map",
        from_json(col("_rescued_data"), MapType(StringType(), StringType()))
    )
    df = df.withColumn("order_id",
        when(
            col("_rescued_data").isNotNull(),
            coalesce(col("Order_id"), col("rescued_map")["Order_ID"])
        ).otherwise(col("Order_id"))
    ).withColumn("product_id",
        when(
            col("_rescued_data").isNotNull(),
            coalesce(col("Product_id"), col("rescued_map")["Product_ID"])
        ).otherwise(col("Product_id"))
    ).drop("rescued_map", "Order_id", "Product_id")

    # ── Step 2: Discount normalisation ────────────────────────────────────
    # Region 4 uses "40%" string format; all other regions use decimal "0.4".
    # Detect "%" suffix → strip it → divide by 100 → consistent 0–1 range.
    df = df.withColumn("discount",
        when(
            col("discount").rlike(r"%$"),
            regexp_replace(col("discount"), "%", "").cast(DoubleType()) / 100
        ).otherwise(col("discount").cast(DoubleType()))
    )

    # ── Step 3: Cast remaining string columns to correct numeric types ─────
    df = df.withColumn("Sales",               col("Sales").cast(DoubleType())) \
           .withColumn("Quantity",             col("Quantity").cast("int")) \
           .withColumn("payment_installments", col("payment_installments").cast("int"))

    # ── Step 4: Lowercase canonical column aliases ─────────────────────────
    # Bronze has mixed casing (Sales, Quantity). Normalise to lowercase so
    # Silver rules are unambiguous.
    df = df.withColumnRenamed("Sales",    "sales") \
           .withColumnRenamed("Quantity", "quantity")

    return df